[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C14_DL_Theory_Data_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零复现** DL 理论与数据/生成评测的核心现象，再用 `assert` **钉死**每条结论。

这个 notebook 做四件事：① 确认环境；② 用一个 60 行的迷你例子**预演**全课五个现象各自的「最小内核」；③ 立下全课纪律——**可复现 + 对拍自检**；④ 给出一个贯穿全课的 `check` 工具。

## 1 · 环境自检

只需要 `numpy`。`matplotlib`/`pandas` 可选。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
for opt in ['matplotlib', 'pandas']:
    try:
        m = __import__(opt); print(opt, getattr(m, '__version__', '?'), '(可选)')
    except Exception:
        print(opt, '未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 可复现性：固定种子是研究的底线

本课每个数字都要**可逐位复现**。统一用 `np.random.default_rng(seed)`（新式 Generator，比全局 `np.random.seed` 更干净、可并行）。

下面验证：**同种子两次采样完全相同；不同种子不同**。这是后面所有「曲线可复现」的前提。

In [ ]:
def draw(seed, n=5):
    rng = np.random.default_rng(seed)
    return rng.standard_normal(n)

a1 = draw(0); a2 = draw(0); b = draw(1)
print('seed=0  第1次:', np.round(a1, 4))
print('seed=0  第2次:', np.round(a2, 4))
print('seed=1       :', np.round(b, 4))
assert np.array_equal(a1, a2), '同种子必须逐位相同（可复现）'
assert not np.array_equal(a1, b), '不同种子应不同'
print('✅ 可复现性成立：同种子=逐位相同，不同种子=不同')

## 3 · 预演现象一：双下降的最小内核（最小范数解）

模块 01 的核心是：**过参数化区，优化器隐式偏向最小范数插值解**。这里用最小例子点一下——
对一个**欠定**线性系统 `Xw=y`（方程少、未知多，有无穷多解），`np.linalg.lstsq` 给出的正是**范数最小**的那个解。

In [ ]:
rng = np.random.default_rng(0)
n, d = 5, 20                      # 5 个方程, 20 个未知 -> 过参数化, 无穷多插值解
X = rng.standard_normal((n, d))
y = rng.standard_normal(n)

w_minnorm, *_ = np.linalg.lstsq(X, y, rcond=None)   # 最小范数解
# 构造另一个也能插值的解：w_minnorm + 任意零空间方向
U, S, Vt = np.linalg.svd(X)
null_dir = Vt[-1]                 # X 的零空间一个方向 (Xv=0)
w_other = w_minnorm + 3.0 * null_dir

print('两个解都插值吗？')
print('  ||X w_minnorm - y|| =', np.linalg.norm(X @ w_minnorm - y))
print('  ||X w_other   - y|| =', np.linalg.norm(X @ w_other   - y))
print(f'范数: 最小范数解 {np.linalg.norm(w_minnorm):.3f}  vs 另一解 {np.linalg.norm(w_other):.3f}')
assert np.allclose(X @ w_minnorm, y, atol=1e-8), '最小范数解应插值'
assert np.allclose(X @ w_other,   y, atol=1e-8), '加零空间方向后仍插值'
assert np.linalg.norm(w_minnorm) < np.linalg.norm(w_other), '最小范数解范数应最小'
print('✅ 过参数化下有无穷多插值解，lstsq 选了范数最小的那个 —— 这就是双下降第二段的种子')

## 4 · 预演现象二：grokking 的最小内核（记忆 vs 泛化）

模块 02 的核心是：**模型可以先记住训练集、很久后才学到泛化规律**。这里用最小例子点一下——
一个查表式「记忆器」在训练集上 100% 正确，但在测试集上等于瞎猜；而一个学到**真规律**的模型两边都对。grokking 就是从前者**过渡到**后者。

In [ ]:
p = 7
# 任务: (a+b) mod p。全部 pairs
pairs = [(a, b) for a in range(p) for b in range(p)]
labels = {(a, b): (a + b) % p for (a, b) in pairs}
train = pairs[:30]; test = pairs[30:]

# 记忆器: 只背训练集, 没见过的瞎猜(返回0)
memo = {ab: labels[ab] for ab in train}
def memorizer(ab): return memo.get(ab, 0)
# 泛化器: 学到了真规律
def generalizer(ab): return (ab[0] + ab[1]) % p

def acc(fn, data): return np.mean([fn(ab) == labels[ab] for ab in data])
print(f'记忆器:  train acc={acc(memorizer, train):.2f}  test acc={acc(memorizer, test):.2f}')
print(f'泛化器:  train acc={acc(generalizer, train):.2f}  test acc={acc(generalizer, test):.2f}')
assert acc(memorizer, train) == 1.0 and acc(memorizer, test) < 0.5, '记忆器: 训练满分、测试瞎猜'
assert acc(generalizer, test) == 1.0, '泛化器: 测试也满分'
print('✅ 记忆与泛化是两种解；grokking = 训练动力学从「记忆」缓慢挪向「泛化」(模块02 复现全过程)')

## 5 · 预演现象三：度量决定曲线形状（涌现假象）

模块 02 还会复现 Schaeffer 的核心论点：**同一个平滑提升的底层能力，用不连续度量看是「涌现跳变」，用连续度量看是「平滑增长」**。

玩具：某能力 = 模型每一步正确的概率 `pc` 随规模平滑上升。任务要 5 步**全对**才算成功（exact-match，不连续）；而单步正确率是连续的。

In [ ]:
scales = np.linspace(0, 1, 11)          # 规模(归一化)
p_correct = scales ** 1.5               # 单步正确率: 平滑上升
k = 5                                   # 要连续 k 步全对
exact_match = p_correct ** k            # all-or-nothing 度量

print(f"{'规模':>6}{'单步正确率(连续)':>16}{'5步全对(不连续)':>18}")
for s, pc, em_ in zip(scales, p_correct, exact_match):
    print(f'{s:>6.1f}{pc:>16.3f}{em_:>18.4f}')

# 连续度量: 近似线性/平滑; 不连续度量: 长期贴地、末端骤升(像涌现)
lo, hi = exact_match[5], exact_match[-1]
assert p_correct[5] > 0.3, '连续度量中段已明显非零(平滑)'
assert exact_match[5] < 0.1 and hi > 0.5, 'exact-match 中段仍贴地、末端才骤升(像跳变)'
print('\n✅ 同一平滑能力, exact-match 让它看起来「涌现」—— 度量的选择制造了表观跳变')

## 6 · 预演现象四：自训练导致方差收缩（model collapse）

模块 04 的核心：**反复用「上一代有限样本的估计」去生成下一代，方差会系统性收缩**。

最小内核：从一个高斯采 N 个点 → 用样本估计均值/方差 → 用估计的高斯再采 N 个点 → 重复。看方差怎么一代代变小。

In [ ]:
def collapse_one_chain(gen0_std=1.0, N=10, generations=8, seed=0):
    rng = np.random.default_rng(seed)
    mu, var = 0.0, gen0_std ** 2
    history = [var]
    for _ in range(generations):
        sample = rng.normal(mu, np.sqrt(max(var, 0)), size=N)   # 用当前估计的分布采样
        mu = sample.mean()                                       # 重新估计(只有 N 个样本)
        var = sample.var()                                       # 有偏估计, 系统性偏小
        history.append(var)
    return history

# 多链平均, 抚平单链噪声; N 小(每代样本少) -> 坍塌更快更明显
chains = np.array([collapse_one_chain(N=10, generations=8, seed=s) for s in range(500)])
mean_var = chains.mean(axis=0)
print('各代平均方差:', np.round(mean_var, 3))
assert mean_var[0] == 1.0
assert mean_var[-1] < mean_var[0] * 0.7, '若干代后方差应明显收缩'
assert np.all(np.diff(mean_var) <= 1e-9), '方差应单调(几乎)不增 -> 逐代收缩'
print('✅ 递归自训练 -> 方差逐代收缩 -> 分布坍塌(模块04 量化几何衰减并给出解药)')

## 7 · 预演现象五：FID 的最小内核（两高斯的距离）

模块 05 的核心：**把真实与生成特征各拟合一个高斯，算它们之间的 Fréchet 距离**。

一维最小例子（高维同理，只是协方差变矩阵）：两个一维高斯的 Fréchet 距离有闭式 `(μ1−μ2)² + (σ1−σ2)²`。验证：同分布距离为 0，越远越大。

In [ ]:
def fid_1d(mu1, s1, mu2, s2):
    '''一维高斯间 Fréchet 距离: (μ1-μ2)^2 + (σ1-σ2)^2 (协方差是标量时 sqrt(Σ1Σ2)=σ1σ2)。'''
    return (mu1 - mu2) ** 2 + (s1 - s2) ** 2

print('FID(同分布)        =', fid_1d(0, 1, 0, 1))
print('FID(均值差 2)      =', fid_1d(0, 1, 2, 1))
print('FID(方差也不同)    =', fid_1d(0, 1, 2, 3))
assert fid_1d(0, 1, 0, 1) == 0.0, '同分布 FID 必须为 0'
assert fid_1d(0, 1, 2, 1) > 0 and fid_1d(0, 1, 2, 3) > fid_1d(0, 1, 2, 1), '越远越大'
print('✅ FID = 均值差(保真) + 协方差差(多样/结构); 模块05 推广到高维并从零算矩阵平方根')

## 8 · 一个贯穿全课的自检工具

把「断言一个数值关系成立并打印」封装成 `check`，后面每个模块都用它当统一裁判。

In [ ]:
def check(name, cond, detail=''):
    '''统一自检: cond 为真则打印通过, 否则 AssertionError。'''
    status = '✅' if cond else '❌'
    print(f'[{status}] {name}' + (f'  | {detail}' if detail else ''))
    assert cond, f'自检失败: {name}'
    return cond

check('环境可复现', np.array_equal(draw(7), draw(7)))
check('双下降种子: 最小范数解范数最小', np.linalg.norm(w_minnorm) < np.linalg.norm(w_other))
check('FID 自反性', fid_1d(0,1,0,1) == 0.0, 'FID(X,X)=0')
print('\n这就是全课的工作流：复现现象 -> 用 check/assert 钉死该成立的关系。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里复现的每个现象，都会用 `assert` 钉死「它确实按预言发生」；可复现 + 可验证，才敢说「我理解了这个机制」。

**接下来五个内容模块**：01 双下降 → 02 grokking/涌现 → 03 数据流水线 → 04 合成数据坍塌 → 05 生成媒体评测。

下一站：**模块 01 · 泛化与双下降**。